# LeetCode 751 — IP to CIDR bằng Python

Link đề: https://leetcode.com/problems/ip-to-cidr/ · Độ khó: **Medium** · Chủ đề: Bit Manipulation, String, Greedy

> **Bài Premium (🔒).** Đề trong notebook này được viết theo hiểu biết về bài (kèm các ví dụ trong tài liệu tham khảo); nếu bạn có tài khoản Premium, hãy đối chiếu lại đề gốc trên LeetCode.

Skeleton LeetCode:

```python
class Solution:
    def ipToCIDR(self, ip: str, n: int) -> List[str]:
        ...
```

Ghi chú: bài có **nhiều đáp án đúng** ("return any of them": thứ tự liệt kê, hoặc cách viết địa chỉ cơ sở của một khối, có thể khác nhau), nên mọi test dùng hàm kiểm tra `check_cidr` (xác nhận các khối **phủ đúng và chỉ đúng** dải IP, không chồng lấn, và số khối là **tối thiểu** theo một oracle quy hoạch động độc lập), không so cứng với chuỗi mẫu.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho địa chỉ IPv4 bắt đầu `ip` và số lượng `n`, phủ **chính xác** dải `[ip, ip + n - 1]` bằng **ít khối CIDR nhất** (mỗi khối là `địa_chỉ/độ_dài_tiền_tố`).
2. Nắm các công cụ Python: đổi IP `"a.b.c.d"` ↔ số nguyên 32 bit bằng `split`, dịch bit `<<`, `>>`, và `&`; lấy **bit thấp nhất** `x & -x`; tìm lũy thừa của 2 lớn nhất không vượt `n` bằng `bit_length`; định dạng chuỗi bằng f-string.
3. Đi từ **mỗi IP một khối `/32`** (đúng nhưng không tối thiểu) → **tham lam**: tại mỗi bước lấy khối **lớn nhất** vừa **căn đúng biên** vừa **không vượt phần còn lại** (lời giải nộp) → **quy hoạch động** (số khối tối thiểu, làm oracle) → **đệ quy trên cây nhị phân tiền tố** (kiểu segment tree).
4. Hiểu mấu chốt: khối CIDR `/k` là một đoạn **liên tiếp** gồm `2^(32-k)` địa chỉ và **bắt đầu tại bội của `2^(32-k)`** (căn biên). Khối lớn nhất bắt đầu tại `x` có kích thước `min(bit thấp nhất của x, lũy thừa 2 lớn nhất ≤ n còn lại)`.
5. Tránh các lỗi kinh điển: chỉ giới hạn theo căn biên mà quên phần còn lại (vượt dải), chỉ giới hạn theo phần còn lại mà quên căn biên, tính sai độ dài tiền tố lệch 1, ghép lại IP thiếu `& 255`, và `x & -x` bằng `0` khi `x = 0` gây lặp vô hạn.

## 2. Đề bài & ví dụ

Một **địa chỉ IP** (IPv4) là một số nguyên 32 bit không dấu, in thành 4 số thập phân (mỗi số là 8 bit) ngăn bởi dấu chấm. Ví dụ nhị phân `00001111 10001000 11111111 01101011` in thành `"15.136.255.107"`.

Một **khối CIDR** là chuỗi gồm một IP cơ sở, dấu `/`, và độ dài tiền tố `k`. Nó phủ **mọi địa chỉ có `k` bit đầu giống `k` bit đầu của IP cơ sở**. Ví dụ `"123.45.67.89/20"` phủ mọi địa chỉ có dạng nhị phân `01111011 00101101 0100xxxx xxxxxxxx` (`x` là 0 hoặc 1).

Cho IP bắt đầu `ip` và số lượng `n`. Hãy dùng **ít khối CIDR nhất** để phủ **chính xác** các địa chỉ trong đoạn đóng `[ip, ip + n - 1]`: không được phủ địa chỉ nào ngoài đoạn đó. Trả về danh sách khối CIDR ngắn nhất; nếu có nhiều đáp án, trả về **bất kỳ**.

**Ví dụ 1**

- Input: `ip = "255.0.0.7"`, `n = 10`
- Output: `["255.0.0.7/32","255.0.0.8/29","255.0.0.16/32"]`
- Giải thích: cần phủ `255.0.0.7` đến `255.0.0.16`. `/32` phủ đúng một địa chỉ (`.7`), `/29` phủ 8 địa chỉ `.8` đến `.15`, và `/32` phủ `.16`. Khối `255.0.0.0/28` phủ cả đoạn nhưng lấn ra ngoài (`.0` đến `.6`), nên không được dùng.

**Ví dụ 2**

- Input: `ip = "117.145.102.62"`, `n = 8`
- Output: `["117.145.102.62/31","117.145.102.64/30","117.145.102.68/31"]`

**Ràng buộc:**

- `7 <= ip.length <= 15`; `ip` là IPv4 hợp lệ dạng `"a.b.c.d"` với `a, b, c, d` trong `[0, 255]`.
- `1 <= n <= 1000`.
- Mọi địa chỉ `ip + x` (với `x < n`) đều là IPv4 hợp lệ.

Chú ý: dải `n` địa chỉ liên tiếp có thể băng qua ranh giới giữa các byte (ví dụ `.255` rồi sang `.0` của byte kế tiếp), nên cách đáng tin cậy là **đổi IP sang số nguyên**, tính toán trên số, rồi đổi lại thành chuỗi.

In [1]:
from typing import List
import random
import time

examples = [
    ("255.0.0.7", 10, ["255.0.0.7/32", "255.0.0.8/29", "255.0.0.16/32"]),
    ("117.145.102.62", 8, ["117.145.102.62/31", "117.145.102.64/30", "117.145.102.68/31"]),
]
for ip, n, expected in examples:
    print(f"ip={ip!r}, n={n} -> đáp án mẫu {expected}")

ip='255.0.0.7', n=10 -> đáp án mẫu ['255.0.0.7/32', '255.0.0.8/29', '255.0.0.16/32']
ip='117.145.102.62', n=8 -> đáp án mẫu ['117.145.102.62/31', '117.145.102.64/30', '117.145.102.68/31']


## 3. Kiến thức nền

### 3.1. IP là một số nguyên 32 bit: `split`, `<<`, `|`

`"a.b.c.d"` ứng với số `a·2^24 + b·2^16 + c·2^8 + d`. Trong Python: tách bằng `ip.split('.')`, đổi từng phần sang `int`, rồi **dịch trái** `a << 24` (nhân `2^24`), `b << 16`, `c << 8`, và ghép bằng `|` (hoặc `+`). Nhờ đó "`ip + 1`" nghĩa là địa chỉ kế tiếp một cách tự nhiên, kể cả khi vượt qua byte.

In [2]:
def ip_to_int(ip):
    a, b, c, d = map(int, ip.split("."))
    return (a << 24) | (b << 16) | (c << 8) | d

print(ip_to_int("0.0.0.1"), ip_to_int("0.0.1.0"), ip_to_int("1.0.0.0"), ip_to_int("255.255.255.255"))
x = ip_to_int("15.136.255.107")
print("15.136.255.107 ->", x, "=", format(x, "032b"))
print("2**32 - 1 =", 2 ** 32 - 1)

1 256 16777216 4294967295
15.136.255.107 -> 260636523 = 00001111100010001111111101101011
2**32 - 1 = 4294967295


### 3.2. Đổi số về IP: dịch phải `>>` và mặt nạ `& 255`

Để lấy lại từng byte, dịch phải rồi giữ **8 bit thấp** bằng `& 255` (vì `255 = 0b11111111`). Byte cao nhất là `x >> 24`, kế đó `(x >> 16) & 255`, `(x >> 8) & 255`, và `x & 255`. Nếu quên `& 255` ở các byte giữa, bạn sẽ in ra số lớn hơn `255`.

In [3]:
def int_to_ip(x):
    return ".".join(str((x >> shift) & 255) for shift in (24, 16, 8, 0))

for ip in ("15.136.255.107", "255.0.0.7", "0.0.0.0", "255.255.255.255"):
    assert int_to_ip(ip_to_int(ip)) == ip
print(int_to_ip(ip_to_int("255.0.0.7") + 10))             # nhảy qua ranh giới? .17
print(int_to_ip(ip_to_int("1.2.3.255") + 1))               # .255 + 1 -> byte kế tiếp tăng 1, byte cuối về 0
print("không & 255:", ".".join(str(ip_to_int("1.2.3.4") >> s) for s in (24, 16, 8, 0)), "(sai: số lớn hơn 255)")

255.0.0.17
1.2.4.0
không & 255: 1.258.66051.16909060 (sai: số lớn hơn 255)


### 3.3. Khối CIDR: độ dài tiền tố `k`, kích thước `2^(32-k)`, và **căn biên**

Khối `/k` cố định `k` bit đầu, `32 - k` bit cuối **tự do** nên phủ `2^(32-k)` địa chỉ **liên tiếp**. Các địa chỉ đó là `base, base + 1, ..., base + 2^(32-k) - 1` với `base` có `32 - k` bit cuối đều là `0`: tức `base` là **bội của `2^(32-k)`** (căn biên, *aligned*). Hệ quả: khối `/32` có 1 địa chỉ, `/31` có 2, `/30` có 4, `/29` có 8, ..., `/24` có 256, `/0` có `2^32` (mọi địa chỉ).

Hàm dưới tính đoạn `[đầu, cuối]` mà một chuỗi khối phủ, kể cả khi IP cơ sở **không căn biên** (khi đó các bit thấp tự do bị bỏ qua).

In [4]:
def parse_cidr(s):
    ip, k = s.split("/")
    return ip_to_int(ip), int(k)

def block_range(s):
    base, k = parse_cidr(s)
    size = 1 << (32 - k)
    start = (base >> (32 - k)) << (32 - k)      # xóa 32-k bit thấp -> đầu khối (căn biên)
    return start, start + size - 1

for s in ["255.0.0.8/29", "255.0.0.7/32", "123.45.67.89/20", "0.0.0.0/0", "10.0.0.0/24"]:
    lo, hi = block_range(s)
    print(f"{s:18s} phủ {int_to_ip(lo)} .. {int_to_ip(hi)}  ({hi - lo + 1} địa chỉ)")
print("255.0.0.9/29 cũng là khối đó:", block_range("255.0.0.9/29") == block_range("255.0.0.8/29"))

255.0.0.8/29       phủ 255.0.0.8 .. 255.0.0.15  (8 địa chỉ)
255.0.0.7/32       phủ 255.0.0.7 .. 255.0.0.7  (1 địa chỉ)
123.45.67.89/20    phủ 123.45.64.0 .. 123.45.79.255  (4096 địa chỉ)
0.0.0.0/0          phủ 0.0.0.0 .. 255.255.255.255  (4294967296 địa chỉ)
10.0.0.0/24        phủ 10.0.0.0 .. 10.0.0.255  (256 địa chỉ)
255.0.0.9/29 cũng là khối đó: True


Dòng cuối cho thấy vì sao đề nói "return any": **cùng một khối** có thể được viết bằng nhiều IP cơ sở khác nhau (`255.0.0.8/29` và `255.0.0.9/29` phủ cùng 8 địa chỉ). Hàm kiểm tra của ta xử lý được cả những cách viết này.

### 3.4. Bit thấp nhất `x & -x`

Với số nguyên `x > 0`, `x & -x` giữ lại **duy nhất bit `1` thấp nhất** của `x` (nhờ biểu diễn bù hai: `-x` đảo mọi bit rồi cộng 1). Giá trị đó chính là **lũy thừa của 2 lớn nhất chia hết `x`**, tức kích thước khối căn biên lớn nhất bắt đầu tại `x`. Ví dụ `x = 40 = 101000` thì `x & -x = 8`: `40` là bội của `8` nhưng không phải bội của `16`.

Cẩn thận `x = 0`: `0 & -0 = 0` (không có bit `1` nào) trong khi thực tế `0` là bội của **mọi** lũy thừa của 2; ta phải xử lý riêng.

In [5]:
for x in (1, 6, 7, 8, 12, 40, 64, 96):
    print(f"x = {x:3d} = {x:08b} | x & -x = {x & -x:3d} -> khối căn biên lớn nhất bắt đầu tại x có {x & -x} địa chỉ")
print("0 & -0 =", 0 & -0, "(cần xử lý riêng)")
print("(-6) =", bin(-6 & 0xFF), "(8 bit thấp của -6 trong bù hai)")

x =   1 = 00000001 | x & -x =   1 -> khối căn biên lớn nhất bắt đầu tại x có 1 địa chỉ
x =   6 = 00000110 | x & -x =   2 -> khối căn biên lớn nhất bắt đầu tại x có 2 địa chỉ
x =   7 = 00000111 | x & -x =   1 -> khối căn biên lớn nhất bắt đầu tại x có 1 địa chỉ
x =   8 = 00001000 | x & -x =   8 -> khối căn biên lớn nhất bắt đầu tại x có 8 địa chỉ
x =  12 = 00001100 | x & -x =   4 -> khối căn biên lớn nhất bắt đầu tại x có 4 địa chỉ
x =  40 = 00101000 | x & -x =   8 -> khối căn biên lớn nhất bắt đầu tại x có 8 địa chỉ
x =  64 = 01000000 | x & -x =  64 -> khối căn biên lớn nhất bắt đầu tại x có 64 địa chỉ
x =  96 = 01100000 | x & -x =  32 -> khối căn biên lớn nhất bắt đầu tại x có 32 địa chỉ
0 & -0 = 0 (cần xử lý riêng)
(-6) = 0b11111010 (8 bit thấp của -6 trong bù hai)


### 3.5. Lũy thừa của 2 lớn nhất không vượt `n`: `bit_length`

`n.bit_length()` là số bit cần để biểu diễn `n` (ví dụ `10 = 1010` có 4 bit). Do đó `1 << (n.bit_length() - 1)` là lũy thừa của 2 **lớn nhất không vượt `n`**: `n = 10` cho `8`, `n = 9` cho `8`, `n = 16` cho `16`, `n = 1` cho `1`. Số bit tối đa của kích thước khối `s = 2^j` chính là `j = s.bit_length() - 1`, từ đó độ dài tiền tố là `32 - j`.

In [6]:
for n in (1, 2, 3, 8, 9, 10, 16, 1000):
    p = 1 << (n.bit_length() - 1)
    print(f"n = {n:4d} | bit_length = {n.bit_length():2d} | lũy thừa 2 lớn nhất ≤ n: {p:4d} = 2^{p.bit_length() - 1}")
size = 8
print("khối 8 địa chỉ = 2^3 -> tiền tố /", 32 - (size.bit_length() - 1))

n =    1 | bit_length =  1 | lũy thừa 2 lớn nhất ≤ n:    1 = 2^0
n =    2 | bit_length =  2 | lũy thừa 2 lớn nhất ≤ n:    2 = 2^1
n =    3 | bit_length =  2 | lũy thừa 2 lớn nhất ≤ n:    2 = 2^1
n =    8 | bit_length =  4 | lũy thừa 2 lớn nhất ≤ n:    8 = 2^3
n =    9 | bit_length =  4 | lũy thừa 2 lớn nhất ≤ n:    8 = 2^3
n =   10 | bit_length =  4 | lũy thừa 2 lớn nhất ≤ n:    8 = 2^3
n =   16 | bit_length =  5 | lũy thừa 2 lớn nhất ≤ n:   16 = 2^4
n = 1000 | bit_length = 10 | lũy thừa 2 lớn nhất ≤ n:  512 = 2^9
khối 8 địa chỉ = 2^3 -> tiền tố / 29


### 3.6. Định dạng chuỗi bằng f-string

Để tạo chuỗi `"255.0.0.8/29"` ta ghép `int_to_ip(x)` với `/` và độ dài tiền tố: `f"{int_to_ip(x)}/{k}"`. f-string chèn giá trị biểu thức vào trong `{}`.

In [7]:
x, k = ip_to_int("255.0.0.8"), 29
print(f"{int_to_ip(x)}/{k}")
print(["{}/{}".format(int_to_ip(x + i), 32) for i in range(3)])

255.0.0.8/29
['255.0.0.8/32', '255.0.0.9/32', '255.0.0.10/32']


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 1: `ip = 255.0.0.7`, `n = 10`, tức cần phủ các địa chỉ có byte cuối từ `7` đến `16` (byte đầu `255.0.0` giữ nguyên). Viết nhị phân 5 bit thấp:

| byte cuối | 7 | 8 | 9 | 10 | 11 | 12 | 13 | 14 | 15 | 16 |
|---|---|---|---|---|---|---|---|---|---|---|
| nhị phân | `00111` | `01000` | `01001` | `01010` | `01011` | `01100` | `01101` | `01110` | `01111` | `10000` |

Quan sát:

- `8` đến `15` đều bắt đầu bằng `01` rồi ba bit tự do: **một khối `/29` (8 địa chỉ)** phủ trọn.
- `7` đứng một mình (`00111`): nếu muốn gộp `7` với `6` hay `4..7` thì phải phủ cả `4, 5, 6` nằm ngoài đoạn; không được. Nên `7` cần khối `/32`.
- `16` đứng một mình ở cuối: khối kế (`16..17`) sẽ lấn sang `17`, ngoài đoạn. Nên `16` cần `/32`.

Cách ngây thơ nhất luôn đúng: dùng **một khối `/32` cho mỗi địa chỉ** (10 khối). Nhưng đề đòi **ít nhất**, và đáp án tối thiểu chỉ có 3 khối. Muốn ít khối ta phải dùng khối **to nhất có thể** tại mỗi vị trí. Hãy xem trực tiếp: với mỗi `x`, các kích thước khối hợp lệ là những lũy thừa của 2 mà (a) `x` chia hết (căn biên) và (b) vừa trong phần còn lại.

In [8]:
def valid_sizes(x, remaining):
    """Các kích thước 2^j (j = 0..32) mà khối bắt đầu tại x hợp lệ: căn biên và không vượt phần còn lại."""
    return [1 << j for j in range(33) if x % (1 << j) == 0 and (1 << j) <= remaining]

x = ip_to_int("255.0.0.7")
remaining = 10
while remaining > 0:
    sizes = valid_sizes(x, remaining)
    print(f"{int_to_ip(x):12s} còn {remaining:2d} địa chỉ -> kích thước hợp lệ {sizes} -> lấy lớn nhất {max(sizes)}")
    x += max(sizes)
    remaining -= max(sizes)

255.0.0.7    còn 10 địa chỉ -> kích thước hợp lệ [1] -> lấy lớn nhất 1
255.0.0.8    còn  9 địa chỉ -> kích thước hợp lệ [1, 2, 4, 8] -> lấy lớn nhất 8
255.0.0.16   còn  1 địa chỉ -> kích thước hợp lệ [1] -> lấy lớn nhất 1


Mỗi lần ta lấy **kích thước hợp lệ lớn nhất**; chỉ ba lần là xong (`1, 8, 1`), khớp đáp án mẫu. Câu hỏi: lấy lớn nhất tại mỗi bước (tham lam) có luôn cho số khối **tối thiểu** không? Ta sẽ chứng minh ở mục 10 và kiểm chứng bằng quy hoạch động ở Cách 3.

## 5. Cách 1 — Baseline: mỗi địa chỉ một khối `/32`

### Ý tưởng

Cách đơn giản nhất mà vẫn **đúng về phủ**: duyệt `n` địa chỉ liên tiếp và in mỗi địa chỉ thành một khối `/32`. Không cần hiểu căn biên hay tiền tố.

### Code

In [9]:
def ip_to_cidr_singles(ip, n):
    x = ip_to_int(ip)
    return [f"{int_to_ip(x + i)}/32" for i in range(n)]

ip, n, expected = examples[0][0], examples[0][1], examples[0][2]
got = ip_to_cidr_singles(ip, n)
print(len(got), "khối:", got)
print("đáp án tối thiểu có", len(expected), "khối")

10 khối: ['255.0.0.7/32', '255.0.0.8/32', '255.0.0.9/32', '255.0.0.10/32', '255.0.0.11/32', '255.0.0.12/32', '255.0.0.13/32', '255.0.0.14/32', '255.0.0.15/32', '255.0.0.16/32']
đáp án tối thiểu có 3 khối


### Độ phức tạp và nút thắt

- **Time:** `O(n)`; **Space:** `O(n)` (kết quả dài `n` khối).
- Nhưng kết quả **không phải ngắn nhất**: 10 khối so với 3. Ở ví dụ 2 ta cũng được 8 khối thay vì 3.

Nút thắt: ta chưa tận dụng được việc **một khối `/k` phủ cả đoạn `2^(32-k)` địa chỉ**. Ta cần một quy tắc để gộp các địa chỉ liên tiếp thành khối lớn khi (và chỉ khi) chúng tạo thành một khối căn biên nằm trọn trong dải. Trước hết ta viết **hàm kiểm tra** (dùng cho mọi test) để thấy rõ cách này "đúng phủ nhưng không tối thiểu".

### Hàm kiểm tra `check_cidr` (dùng xuyên suốt notebook)

Vì có nhiều đáp án đúng, ta cần kiểm tra **tính chất**:

1. Mỗi chuỗi có dạng `ip/k` hợp lệ với `0 <= k <= 32`.
2. Các khối (sau khi đổi thành đoạn `[đầu, cuối]` và sắp theo đầu) **nối tiếp nhau không chồng lấn và không hở**, bắt đầu đúng tại `ip` và kết thúc đúng tại `ip + n - 1`. Tức phủ **chính xác** dải.
3. Số khối bằng **số khối tối thiểu** do một oracle quy hoạch động độc lập tính ra (`min_blocks_dp`, mô tả ở Cách 3).

Dưới đây là phiên bản tổng quát cho độ rộng `bits` bit (mặc định `32`) để tái sử dụng khi thử với miền nhỏ.

In [10]:
def min_blocks_dp(start, n, bits=32):
    """Số khối CIDR tối thiểu để phủ chính xác [start, start+n-1] (quy hoạch động, độc lập với tham lam)."""
    INF = float("inf")
    f = [INF] * (n + 1)
    f[n] = 0
    for t in range(n - 1, -1, -1):
        pos = start + t
        for j in range(bits + 1):
            size = 1 << j
            if size > n - t:
                break
            if pos % size == 0 and f[t + size] + 1 < f[t]:
                f[t] = f[t + size] + 1
    return f[0]

def block_range_bits(base, prefix, bits=32):
    size = 1 << (bits - prefix)
    lo = (base >> (bits - prefix)) << (bits - prefix)
    return lo, lo + size - 1

def check_blocks(start, n, blocks, bits=32, require_min=True):
    """blocks: list các (base, prefix). Kiểm tra phủ chính xác, không chồng lấn, và (tuỳ chọn) tối thiểu."""
    intervals = []
    for base, prefix in blocks:
        assert 0 <= prefix <= bits, "độ dài tiền tố ngoài [0, bits]"
        assert 0 <= base < (1 << bits), "địa chỉ ngoài miền"
        intervals.append(block_range_bits(base, prefix, bits))
    intervals.sort()
    cur = start
    for lo, hi in intervals:
        assert lo == cur, f"hở hoặc chồng lấn tại {lo}, mong đợi {cur}"
        cur = hi + 1
    assert cur == start + n, f"kết thúc sai: {cur - 1}, mong đợi {start + n - 1}"
    if require_min:
        assert len(blocks) == min_blocks_dp(start, n, bits), f"không tối thiểu: {len(blocks)} khối"
    return True

def check_cidr(ip, n, answer, require_min=True):
    blocks = [parse_cidr(s) for s in answer]
    return check_blocks(ip_to_int(ip), n, blocks, 32, require_min)

# Cả hai đáp án mẫu của đề phải qua kiểm tra
for ip, n, expected in examples:
    assert check_cidr(ip, n, expected)
print("đáp án mẫu của đề: hợp lệ và tối thiểu")

# /32 từng địa chỉ: phủ đúng nhưng không tối thiểu
ip, n = examples[0][0], examples[0][1]
assert check_cidr(ip, n, ip_to_cidr_singles(ip, n), require_min=False)
try:
    check_cidr(ip, n, ip_to_cidr_singles(ip, n))
except AssertionError as e:
    print("cách /32 từng địa chỉ bị từ chối:", e)

đáp án mẫu của đề: hợp lệ và tối thiểu
cách /32 từng địa chỉ bị từ chối: không tối thiểu: 10 khối


Hàm kiểm tra chấp nhận cả cách viết khác của cùng một khối. Ví dụ ta viết lại đáp án mẫu 1 với IP cơ sở **không căn biên** và đảo thứ tự: vẫn hợp lệ.

In [11]:
alt = ["255.0.0.16/32", "255.0.0.15/29", "255.0.0.7/32"]       # .15/29 cũng là khối 255.0.0.8/29
assert check_cidr("255.0.0.7", 10, alt)
print(alt, "-> hợp lệ (cùng tập địa chỉ, khác cách viết và thứ tự)")

['255.0.0.16/32', '255.0.0.15/29', '255.0.0.7/32'] -> hợp lệ (cùng tập địa chỉ, khác cách viết và thứ tự)


## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** xét địa chỉ hiện tại `x` và số địa chỉ còn lại `n`. Khối bắt đầu tại `x` có kích thước `s = 2^j` phải thỏa **hai điều kiện**:

1. **Căn biên:** `x` là bội của `s` (nếu không, khối `/k` tương ứng sẽ bắt đầu *trước* `x`, phủ địa chỉ ngoài dải).
2. **Không vượt:** `s <= n` (nếu không, khối phủ quá cuối dải).

Điều kiện 1 chặn `s <= x & -x` (bit thấp nhất của `x`, vì `x` chia hết `2^j` đúng khi `j` không quá số bit `0` thấp liên tiếp). Điều kiện 2 chặn `s <= 2^⌊log2 n⌋`. Mọi lũy thừa của 2 không vượt `min` của hai cận đều hợp lệ, nên kích thước lớn nhất là

`s = min(x & -x, 2^⌊log2 n⌋)`,

(với `x = 0` thì cận căn biên là vô hạn, dùng `2^32` thay thế).

**Tham lam:** lấy khối lớn nhất này, tiến `x += s`, giảm `n -= s`, lặp tới khi `n = 0`. Mỗi bước cắt ít nhất một nửa "độ dốc": sau khối căn biên, bit thấp nhất của `x` tăng hoặc `n` giảm xuống dưới một nửa, nên tổng số khối chỉ `O(log)` (tối đa khoảng `2 · 32 = 64`).

Thử với số cho ví dụ 1 (`x = 7, n = 10` trên 5 bit thấp):

In [12]:
def trace_sizes(x, n):
    steps = []
    while n > 0:
        lowbit = (x & -x) or (1 << 32)
        pow2 = 1 << (n.bit_length() - 1)
        s = min(lowbit, pow2)
        steps.append((x, n, lowbit, pow2, s))
        x += s
        n -= s
    return steps

for x, n, lowbit, pow2, s in trace_sizes(7, 10):
    print(f"x={x:3d} n={n:2d} | căn biên tối đa {lowbit:2d} | vừa phần còn lại tối đa {pow2:2d} | chọn s = min = {s}")

x=  7 n=10 | căn biên tối đa  1 | vừa phần còn lại tối đa  8 | chọn s = min = 1
x=  8 n= 9 | căn biên tối đa  8 | vừa phần còn lại tối đa  8 | chọn s = min = 8
x= 16 n= 1 | căn biên tối đa 16 | vừa phần còn lại tối đa  1 | chọn s = min = 1


Đúng ba bước `1, 8, 1`. Điều thú vị: khi `x` bắt đầu ở vị trí "xấu" (nhiều bit `1` thấp) ta phải dùng khối nhỏ để **leo lên** biên lớn; khi gần cuối dải, `n` nhỏ buộc ta **xuống** các khối nhỏ dần. Dãy kích thước vì thế thường **tăng rồi giảm**.

## 7. Các cách tiếp theo

Tổng cộng 4 cách:

1. Mỗi địa chỉ một khối `/32` (baseline, không tối thiểu).
2. **Tham lam: khối lớn nhất căn biên và vừa phần còn lại** (lời giải nộp).
3. **Quy hoạch động** số khối tối thiểu (dùng làm oracle; cũng dựng được đáp án).
4. **Đệ quy trên cây nhị phân tiền tố** (giống segment tree): nút nằm trọn trong dải thì lấy cả nút.

### 7.1. Cách 2 — Tham lam (lời giải nộp)

Giải thích từng đoạn:

1. `x = ip_to_int(ip)`; danh sách kết quả `res = []`.
2. Trong khi `n > 0`: tính `lowbit = x & -x` (nếu `x == 0` thì dùng `1 << 32`), `pow2 = 1 << (n.bit_length() - 1)`, rồi `size = min(lowbit, pow2)`.
3. Độ dài tiền tố `k = 32 - (size.bit_length() - 1)` (vì `size = 2^j` có `j = size.bit_length() - 1`).
4. Thêm `f"{int_to_ip(x)}/{k}"`, rồi `x += size`, `n -= size`.

In [13]:
def ip_to_cidr_greedy(ip, n):
    x = ip_to_int(ip)
    res = []
    while n > 0:
        lowbit = (x & -x) or (1 << 32)               # x = 0: căn biên tuyệt đối
        size = min(lowbit, 1 << (n.bit_length() - 1))
        k = 32 - (size.bit_length() - 1)
        res.append(f"{int_to_ip(x)}/{k}")
        x += size
        n -= size
    return res

for ip, n, expected in examples:
    got = ip_to_cidr_greedy(ip, n)
    print(ip, n, "->", got)
    assert got == expected
    assert check_cidr(ip, n, got)

255.0.0.7 10 -> ['255.0.0.7/32', '255.0.0.8/29', '255.0.0.16/32']
117.145.102.62 8 -> ['117.145.102.62/31', '117.145.102.64/30', '117.145.102.68/31']


In [14]:
class Solution2:
    def ipToCIDR(self, ip: str, n: int) -> List[str]:
        a, b, c, d = map(int, ip.split("."))
        x = (a << 24) | (b << 16) | (c << 8) | d
        res = []
        while n > 0:
            lowbit = (x & -x) or (1 << 32)
            size = min(lowbit, 1 << (n.bit_length() - 1))
            k = 32 - (size.bit_length() - 1)
            res.append(".".join(str((x >> s) & 255) for s in (24, 16, 8, 0)) + "/" + str(k))
            x += size
            n -= size
        return res

for ip, n, expected in examples:
    assert Solution2().ipToCIDR(ip, n) == expected
print("Solution2 OK")

Solution2 OK


**Độ phức tạp Cách 2:** Time `O(B)` với `B` là số khối (tối đa khoảng `2 · 32`), tức `O(log n + 32)`, hằng số nhỏ; Space `O(B)` cho kết quả. (Trên LeetCode với `n <= 1000`, `B` tối đa cỡ 18.)

### 7.2. Cách 3 — Quy hoạch động (oracle độc lập)

Ý tưởng khác hẳn: không đoán "lấy khối to nhất", mà **thử mọi khối hợp lệ** tại từng vị trí và chọn cách cho số khối ít nhất.

Gọi `f[t]` = số khối tối thiểu để phủ chính xác các địa chỉ từ vị trí `start + t` đến hết dải (`n - t` địa chỉ còn lại). Khi `t = n` không còn gì: `f[n] = 0`. Với `t < n`: thử mọi `j` sao cho `size = 2^j <= n - t` và `start + t` chia hết `size`:

`f[t] = 1 + min f[t + size]`.

Đáp án là `f[0]`. Dựng danh sách khối bằng cách nhớ lựa chọn tại mỗi `t`. Độ phức tạp `O(n · log)`: với mỗi `t` thử tối đa `log2(n) + 1` kích thước. Thuật toán này **không dùng** `x & -x` hay tham lam, nên là chuẩn đối chiếu độc lập.

In [15]:
def ip_to_cidr_dp(ip, n):
    start = ip_to_int(ip)
    INF = float("inf")
    f = [INF] * (n + 1)
    choice = [0] * (n + 1)
    f[n] = 0
    for t in range(n - 1, -1, -1):
        pos = start + t
        for j in range(33):
            size = 1 << j
            if size > n - t:
                break
            if pos % size == 0 and f[t + size] + 1 < f[t]:
                f[t] = f[t + size] + 1
                choice[t] = size
    res, t = [], 0
    while t < n:
        size = choice[t]
        res.append(f"{int_to_ip(start + t)}/{32 - (size.bit_length() - 1)}")
        t += size
    return res

for ip, n, expected in examples:
    got = ip_to_cidr_dp(ip, n)
    print(ip, n, "->", got)
    assert check_cidr(ip, n, got)
    assert len(got) == len(expected)

255.0.0.7 10 -> ['255.0.0.7/32', '255.0.0.8/29', '255.0.0.16/32']
117.145.102.62 8 -> ['117.145.102.62/31', '117.145.102.64/30', '117.145.102.68/31']


In [16]:
class Solution3:
    def ipToCIDR(self, ip: str, n: int) -> List[str]:
        a, b, c, d = map(int, ip.split("."))
        start = (a << 24) | (b << 16) | (c << 8) | d
        INF = float("inf")
        f = [INF] * (n + 1)
        choice = [0] * (n + 1)
        f[n] = 0
        for t in range(n - 1, -1, -1):
            pos = start + t
            for j in range(33):
                size = 1 << j
                if size > n - t:
                    break
                if pos % size == 0 and f[t + size] + 1 < f[t]:
                    f[t] = f[t + size] + 1
                    choice[t] = size
        res, t = [], 0
        while t < n:
            size = choice[t]
            x = start + t
            res.append(".".join(str((x >> s) & 255) for s in (24, 16, 8, 0)) + "/" + str(32 - (size.bit_length() - 1)))
            t += size
        return res

for ip, n, expected in examples:
    assert len(Solution3().ipToCIDR(ip, n)) == len(expected)
    assert check_cidr(ip, n, Solution3().ipToCIDR(ip, n))
print("Solution3 OK")

Solution3 OK


**Độ phức tạp Cách 3:** Time `O(n · log n)` (với `n <= 1000` tức vài chục nghìn phép tính); Space `O(n)` cho `f` và `choice`. Chậm hơn tham lam nhưng **không cần chứng minh tham lam**, nên rất hợp làm oracle.

Cũng nhờ DP ta kiểm tra được điều thú vị: số cách phủ **tối thiểu** (theo tập đoạn) là **duy nhất**. Hãy đếm.

In [17]:
def count_optimal_covers(start, n, bits=32):
    """Đếm số cách (theo tập các đoạn) đạt số khối tối thiểu."""
    INF = float("inf")
    f = [INF] * (n + 1)
    ways = [0] * (n + 1)
    f[n], ways[n] = 0, 1
    for t in range(n - 1, -1, -1):
        pos = start + t
        for j in range(bits + 1):
            size = 1 << j
            if size > n - t:
                break
            if pos % size == 0:
                cand = f[t + size] + 1
                if cand < f[t]:
                    f[t], ways[t] = cand, ways[t + size]
                elif cand == f[t]:
                    ways[t] += ways[t + size]
    return ways[0]

rng = random.Random(11)
for _ in range(2000):
    start = rng.randint(0, 2000)
    n = rng.randint(1, 300)
    assert count_optimal_covers(start, n) == 1
print("2000 dải ngẫu nhiên: mỗi dải chỉ có ĐÚNG MỘT cách phủ tối thiểu (theo tập đoạn)")
print("-> các 'đáp án khác nhau' chỉ khác thứ tự liệt kê hoặc cách viết IP cơ sở của cùng một khối")

2000 dải ngẫu nhiên: mỗi dải chỉ có ĐÚNG MỘT cách phủ tối thiểu (theo tập đoạn)
-> các 'đáp án khác nhau' chỉ khác thứ tự liệt kê hoặc cách viết IP cơ sở của cùng một khối


### 7.3. Cách 4 — Đệ quy trên cây nhị phân tiền tố (kiểu segment tree)

Nhìn không gian địa chỉ như một **cây nhị phân** độ sâu 32: nút gốc là khối `/0` (mọi địa chỉ); mỗi nút khối `/k` có hai con `/k+1` (nửa dưới và nửa trên). Khối CIDR chính là **một nút** của cây này. Để phủ chính xác `[lo, hi]` ta duyệt từ gốc:

- Nếu đoạn của nút **không giao** `[lo, hi]`: bỏ qua.
- Nếu đoạn của nút **nằm trọn** trong `[lo, hi]`: **lấy cả nút** (một khối), không đi xuống nữa.
- Ngược lại (giao một phần): đệ quy hai con.

Đây chính là phép phân rã đoạn thành các nút của segment tree. Thứ tự duyệt trái trước phải sau cho danh sách tăng dần theo địa chỉ. Số nút thăm là `O(B · 32)`.

In [18]:
def cidr_trie_bits(start, n, bits=32):
    lo, hi = start, start + n - 1
    res = []
    def go(node_start, level):                       # nút phủ [node_start, node_start + 2^level - 1]
        node_end = node_start + (1 << level) - 1
        if node_end < lo or node_start > hi:
            return
        if lo <= node_start and node_end <= hi:
            res.append((node_start, bits - level))
            return
        half = 1 << (level - 1)
        go(node_start, level - 1)
        go(node_start + half, level - 1)
    go(0, bits)
    return res

def ip_to_cidr_trie(ip, n):
    return [f"{int_to_ip(b)}/{k}" for b, k in cidr_trie_bits(ip_to_int(ip), n, 32)]

for ip, n, expected in examples:
    got = ip_to_cidr_trie(ip, n)
    print(ip, n, "->", got)
    assert got == expected
    assert check_cidr(ip, n, got)

255.0.0.7 10 -> ['255.0.0.7/32', '255.0.0.8/29', '255.0.0.16/32']
117.145.102.62 8 -> ['117.145.102.62/31', '117.145.102.64/30', '117.145.102.68/31']


In [19]:
class Solution4:
    def ipToCIDR(self, ip: str, n: int) -> List[str]:
        a, b, c, d = map(int, ip.split("."))
        lo = (a << 24) | (b << 16) | (c << 8) | d
        hi = lo + n - 1
        res = []

        def go(node_start: int, level: int) -> None:
            node_end = node_start + (1 << level) - 1
            if node_end < lo or node_start > hi:
                return
            if lo <= node_start and node_end <= hi:
                ip_str = ".".join(str((node_start >> s) & 255) for s in (24, 16, 8, 0))
                res.append(ip_str + "/" + str(32 - level))
                return
            half = 1 << (level - 1)
            go(node_start, level - 1)
            go(node_start + half, level - 1)

        go(0, 32)
        return res

for ip, n, expected in examples:
    assert Solution4().ipToCIDR(ip, n) == expected
print("Solution4 OK")

Solution4 OK


**Độ phức tạp Cách 4:** Time `O(B · 32)` (mỗi khối trên đường đi gồm tối đa vài nút chia nhỏ); Space `O(32)` ngăn xếp đệ quy cộng `O(B)` kết quả. Độ sâu đệ quy chỉ 32, không lo tràn.

Bảng so sánh cả 4 cách ở mục 16.

## 8. TRACE từng bước

Trace Cách 2 (tham lam) trên ví dụ 1: `ip = 255.0.0.7`, `n = 10`.

| bước | `x` (IP) | `x & -x` | `n` còn | lũy thừa 2 lớn nhất `≤ n` | `size` | tiền tố | khối thêm | `x` sau | `n` sau |
|---|---|---|---|---|---|---|---|---|---|
| 1 | `255.0.0.7` | 1 | 10 | 8 | 1 | `/32` | `255.0.0.7/32` | `255.0.0.8` | 9 |
| 2 | `255.0.0.8` | 8 | 9 | 8 | 8 | `/29` | `255.0.0.8/29` | `255.0.0.16` | 1 |
| 3 | `255.0.0.16` | 16 | 1 | 1 | 1 | `/32` | `255.0.0.16/32` | `255.0.0.17` | 0 |

Kết quả 3 khối. Hàm dưới in đúng từng bước, kèm biểu diễn nhị phân.

In [20]:
def trace_greedy(ip, n):
    x = ip_to_int(ip)
    print(f"ip = {ip} = {x}, n = {n}, phủ {ip} .. {int_to_ip(x + n - 1)}")
    res, step = [], 0
    while n > 0:
        step += 1
        lowbit = (x & -x) or (1 << 32)
        pow2 = 1 << (n.bit_length() - 1)
        size = min(lowbit, pow2)
        k = 32 - (size.bit_length() - 1)
        block = f"{int_to_ip(x)}/{k}"
        print(f"bước {step}: x={int_to_ip(x)} (...{x & 0xFFFF:016b}) | lowbit={lowbit if lowbit < (1 << 32) else 'inf'} | n={n} pow2={pow2} | size=min={size} -> /{k} -> {block}")
        res.append(block)
        x += size
        n -= size
    print("kết quả:", res)
    return res

assert trace_greedy("255.0.0.7", 10) == examples[0][2]

ip = 255.0.0.7 = 4278190087, n = 10, phủ 255.0.0.7 .. 255.0.0.16
bước 1: x=255.0.0.7 (...0000000000000111) | lowbit=1 | n=10 pow2=8 | size=min=1 -> /32 -> 255.0.0.7/32
bước 2: x=255.0.0.8 (...0000000000001000) | lowbit=8 | n=9 pow2=8 | size=min=8 -> /29 -> 255.0.0.8/29
bước 3: x=255.0.0.16 (...0000000000010000) | lowbit=16 | n=1 pow2=1 | size=min=1 -> /32 -> 255.0.0.16/32
kết quả: ['255.0.0.7/32', '255.0.0.8/29', '255.0.0.16/32']


Trace ví dụ 2: `ip = 117.145.102.62`, `n = 8`. Byte cuối `62 = 111110`: bit thấp nhất là `2`, nên khối đầu chỉ có `2` địa chỉ (`/31`); sau đó `x = ...64` (bội của `64`), còn `6` địa chỉ nên lấy `4` (`/30`); rồi `x = ...68` (bội của `4`), còn `2` địa chỉ nên lấy `2` (`/31`). Ta thấy dãy kích thước `2, 4, 2` (tăng rồi giảm).

In [21]:
assert trace_greedy("117.145.102.62", 8) == examples[1][2]

ip = 117.145.102.62 = 1972463166, n = 8, phủ 117.145.102.62 .. 117.145.102.69
bước 1: x=117.145.102.62 (...0110011000111110) | lowbit=2 | n=8 pow2=8 | size=min=2 -> /31 -> 117.145.102.62/31
bước 2: x=117.145.102.64 (...0110011001000000) | lowbit=64 | n=6 pow2=4 | size=min=4 -> /30 -> 117.145.102.64/30
bước 3: x=117.145.102.68 (...0110011001000100) | lowbit=4 | n=2 pow2=2 | size=min=2 -> /31 -> 117.145.102.68/31
kết quả: ['117.145.102.62/31', '117.145.102.64/30', '117.145.102.68/31']


Trace Cách 4 (cây tiền tố) trên một miền **nhỏ 6 bit** để dễ đọc: phủ `[7, 16]` (10 địa chỉ) trong miền `0..63`. Mỗi dòng là một nút: `[đầu..cuối]`, và hành động (bỏ qua / lấy cả nút / chia đôi).

In [22]:
def trace_trie_small(start, n, bits=6):
    lo, hi = start, start + n - 1
    res = []
    def go(node_start, level, depth):
        node_end = node_start + (1 << level) - 1
        pad = "  " * depth
        tag = f"nút /{bits - level} [{node_start}..{node_end}]"
        if node_end < lo or node_start > hi:
            print(f"{pad}{tag}: ngoài dải -> bỏ")
            return
        if lo <= node_start and node_end <= hi:
            print(f"{pad}{tag}: nằm trọn trong [{lo}..{hi}] -> LẤY khối {node_start}/{bits - level}")
            res.append((node_start, bits - level))
            return
        print(f"{pad}{tag}: giao một phần -> chia đôi")
        half = 1 << (level - 1)
        go(node_start, level - 1, depth + 1)
        go(node_start + half, level - 1, depth + 1)
    go(0, bits, 0)
    print("kết quả (đầu, tiền tố):", res)
    return res

res = trace_trie_small(7, 10)
assert res == [(7, 6), (8, 3), (16, 6)]

nút /0 [0..63]: giao một phần -> chia đôi
  nút /1 [0..31]: giao một phần -> chia đôi
    nút /2 [0..15]: giao một phần -> chia đôi
      nút /3 [0..7]: giao một phần -> chia đôi
        nút /4 [0..3]: ngoài dải -> bỏ
        nút /4 [4..7]: giao một phần -> chia đôi
          nút /5 [4..5]: ngoài dải -> bỏ
          nút /5 [6..7]: giao một phần -> chia đôi
            nút /6 [6..6]: ngoài dải -> bỏ
            nút /6 [7..7]: nằm trọn trong [7..16] -> LẤY khối 7/6
      nút /3 [8..15]: nằm trọn trong [7..16] -> LẤY khối 8/3
    nút /2 [16..31]: giao một phần -> chia đôi
      nút /3 [16..23]: giao một phần -> chia đôi
        nút /4 [16..19]: giao một phần -> chia đôi
          nút /5 [16..17]: giao một phần -> chia đôi
            nút /6 [16..16]: nằm trọn trong [7..16] -> LẤY khối 16/6
            nút /6 [17..17]: ngoài dải -> bỏ
          nút /5 [18..19]: ngoài dải -> bỏ
        nút /4 [20..23]: ngoài dải -> bỏ
      nút /3 [24..31]: ngoài dải -> bỏ
  nút /1 [32..63]: ngoài dải -> bỏ

Cùng đáp án với tham lam: các khối `7/6` (1 địa chỉ), `8/3` (8 địa chỉ, từ 8 đến 15), `16/6` (1 địa chỉ). Cây cho thấy rõ **vì sao** các khối có hình dạng đó: hai "biên" của dải (7 và 16) mỗi biên sinh ra một chuỗi nút chia đôi, còn phần giữa là các nút lớn nằm trọn.

## 9. Code nộp LeetCode

Ta chọn **Cách 2 (tham lam)**: ngắn, nhanh, và đã được kiểm chứng bằng DP.

In [23]:
class Solution:
    def ipToCIDR(self, ip: str, n: int) -> List[str]:
        a, b, c, d = map(int, ip.split("."))
        x = (a << 24) | (b << 16) | (c << 8) | d
        ans = []
        while n > 0:
            lowbit = (x & -x) or (1 << 32)               # bit thấp nhất: kích thước khối căn biên lớn nhất
            size = min(lowbit, 1 << (n.bit_length() - 1))  # và không vượt phần còn lại
            k = 32 - (size.bit_length() - 1)
            ans.append(f"{(x >> 24) & 255}.{(x >> 16) & 255}.{(x >> 8) & 255}.{x & 255}/{k}")
            x += size
            n -= size
        return ans


sol = Solution()
for ip, n, expected in examples:
    got = sol.ipToCIDR(ip, n)
    print(ip, n, "->", got)
    assert check_cidr(ip, n, got)
    assert got == expected

255.0.0.7 10 -> ['255.0.0.7/32', '255.0.0.8/29', '255.0.0.16/32']
117.145.102.62 8 -> ['117.145.102.62/31', '117.145.102.64/30', '117.145.102.68/31']


## 10. Vì sao đúng

**Khẳng định 1 (khối hợp lệ).** Khối `/k` phủ đúng các địa chỉ có `k` bit đầu bằng IP cơ sở, tức `2^j` địa chỉ liên tiếp (`j = 32 - k`) bắt đầu tại `base` có `j` bit thấp bằng `0`, nghĩa là `base` chia hết `2^j`. Do đó khối có kích thước `s = 2^j` **bắt đầu tại `x`** phủ `[x, x + s - 1]` **khi và chỉ khi** `x` chia hết `s`. (Nếu `x` không chia hết `s` thì khối "chứa `x`" bắt đầu trước `x`, phủ địa chỉ ngoài dải.)

**Khẳng định 2 (kích thước lớn nhất).** `x` chia hết `2^j` khi và chỉ khi `2^j <= x & -x` (đúng với `x > 0`; với `x = 0` mọi `j` đều thỏa). Khối còn phải vừa phần còn lại: `2^j <= n`. Do đó mọi `2^j <= min(x & -x, n)` đều hợp lệ và lũy thừa 2 lớn nhất như thế là `min(x & -x, 2^⌊log2 n⌋)` (vì `x & -x` bản thân đã là lũy thừa của 2).

**Khẳng định 3 (tham lam tối ưu).** Đặt `S = s_max` (đã là lũy thừa của 2, `x` chia hết `S`, `S <= n`). Xét một cách phủ chính xác tối ưu `O` bất kỳ; khối đầu của nó bắt đầu tại `x` và có kích thước `s' <= S`. Nếu `s' = S` thì `O` đã trùng với tham lam ở bước đầu. Nếu `s' < S`: xét mọi khối của `O` bắt đầu trong `[x, x + S)`. Một khối như vậy bắt đầu tại `y` với kích thước `t` (lũy thừa 2, căn biên tại `y`). Nếu `t <= S` thì khối nằm gọn trong khối căn biên cỡ `S` chứa `y`, đó chính là `[x, x + S)` (vì `x` là bội của `S`). Nếu `t > S` thì khối căn biên cỡ `t` chứa `y` phải chứa cả `[x, x + S)` nên bắt đầu tại `y <= x`, tức `y = x` và `t = s' < S`: mâu thuẫn. Vậy mọi khối của `O` bắt đầu trong `[x, x + S)` đều nằm gọn trong đó, và chúng chia đúng `[x, x + S)` (vì `O` phủ chính xác và `S <= n`). Vì khối đầu `s' < S` nên có **ít nhất 2 khối** như thế. Thay chúng bằng **một** khối cỡ `S` cho ra cách phủ hợp lệ với **ít khối hơn**, mâu thuẫn với tính tối ưu của `O`. Vậy mọi cách tối ưu đều bắt đầu bằng khối cỡ `S`, và phần còn lại (`n - S` địa chỉ bắt đầu tại `x + S`) là bài toán con được giải tối ưu theo quy nạp. (Chứng minh cũng cho thấy cách phủ tối thiểu là **duy nhất** theo tập đoạn, khớp thí nghiệm ở mục 7.2.) Ngoài ra DP độc lập ở mục 12 kiểm chứng điều này trên mọi dải nhỏ và hàng nghìn dải ngẫu nhiên.

**Khẳng định 4 (chính xác, không vượt).** Mỗi khối nằm trong `[x, x + n - 1]` còn lại (vì `s <= n`) và bắt đầu đúng tại `x`; sau đó `x += s`, `n -= s` nên các khối **nối tiếp không hở không chồng**; kết thúc khi `n = 0`, tức đúng tại `ip + n_ban_đầu - 1`.

**Khẳng định 5 (hình dạng và số khối).** Phần đầu: bit thấp nhất của `x` tăng sau mỗi khối (nhờ cộng `s = x & -x` làm `x` có nhiều bit `0` thấp hơn) tới khi `n` bắt đầu là cận chặn; phần sau: `n` còn lại giảm ít nhất một nửa mỗi bước (`s` là lũy thừa 2 lớn nhất `<= n`, và `n - s < s`) nên dãy giảm. Mỗi giai đoạn tối đa 32 khối, tổng không quá khoảng `2 · 32`.

**Cách 3 và 4.** DP thử mọi khối hợp lệ nên tối ưu theo định nghĩa. Cây tiền tố: khối CIDR là nút của cây; một nút nằm trọn trong dải thì lấy cả nút (không cần chia nhỏ hơn vì chia nhỏ chỉ thêm khối), nút giao một phần buộc phải chia, nên được phân rã tối thiểu (đúng như phân rã đoạn thành nút segment tree).

## 11. Time / Space Complexity

- **Cách 1 (`/32` mỗi địa chỉ):** Time `O(n)`; Space `O(n)`; **không tối thiểu**.
- **Cách 2 (tham lam, nộp):** Time `O(B)` với `B <= ~64` là số khối (`O(log n)` với `n` thực tế); Space `O(B)`.
- **Cách 3 (DP):** Time `O(n log n)`; Space `O(n)`.
- **Cách 4 (cây tiền tố):** Time `O(B · 32)`; Space `O(32 + B)`.

Hãy đo trên `n = 1000` và vài IP bắt đầu khác nhau, kèm số khối tìm được.

In [24]:
def timeit_once(fn, *args):
    t0 = time.perf_counter()
    res = fn(*args)
    return res, time.perf_counter() - t0

cases = [("0.0.0.0", 1000), ("255.0.0.7", 1000), ("10.20.30.255", 1000), ("255.255.252.24", 1000)]
methods = [("greedy", Solution2().ipToCIDR), ("DP", Solution3().ipToCIDR), ("trie", Solution4().ipToCIDR)]
for ip, n in cases:
    row, sizes = [], set()
    for name, fn in methods:
        res = None
        t0 = time.perf_counter()
        for _ in range(200):
            res = fn(ip, n)
        dt = (time.perf_counter() - t0) / 200
        sizes.add(len(res))
        assert check_cidr(ip, n, res)
        row.append(f"{name}: {dt * 1e6:8.1f} µs")
    print(f"ip={ip:15s} n={n}: số khối = {sizes.pop()} | " + " | ".join(row))

ip=0.0.0.0         n=1000: số khối = 6 | greedy:      6.7 µs | DP:    962.5 µs | trie:     17.9 µs


ip=255.0.0.7       n=1000: số khối = 15 | greedy:     16.5 µs | DP:   1392.9 µs | trie:     54.5 µs


ip=10.20.30.255    n=1000: số khối = 9 | greedy:     17.9 µs | DP:   1356.3 µs | trie:     27.1 µs
ip=255.255.252.24  n=1000: số khối = 6 | greedy:      7.6 µs | DP:    994.3 µs | trie:     18.1 µs


Cả ba cách đều nhanh với `n <= 1000`; tham lam nhanh nhất. DP chậm hơn rõ rệt vì tốn `O(n log n)`, nhưng vẫn chỉ cỡ một phần nghìn giây.

## 12. Bộ test

Bộ test dùng `check_cidr` (phủ chính xác, không chồng lấn, **tối thiểu theo DP**), không so cứng chuỗi. Gồm: ví dụ đề, edge case, phản ví dụ cho các cách "gần đúng", vét cạn các dải nhỏ, và ngẫu nhiên có seed (IP 32 bit đúng ràng buộc).

In [25]:
all_methods = {
    "greedy": ip_to_cidr_greedy,
    "dp": ip_to_cidr_dp,
    "trie": ip_to_cidr_trie,
    "Sol2": lambda ip, n: Solution2().ipToCIDR(ip, n),
    "Sol3": lambda ip, n: Solution3().ipToCIDR(ip, n),
    "Sol4": lambda ip, n: Solution4().ipToCIDR(ip, n),
    "submit": lambda ip, n: Solution().ipToCIDR(ip, n),
}

def run_all(ip, n):
    for name, fn in all_methods.items():
        ans = fn(ip, n)
        assert check_cidr(ip, n, ans), (name, ip, n, ans)
    return all_methods["submit"](ip, n)

for ip, n, expected in examples:
    got = run_all(ip, n)
    assert got == expected            # đáp án mẫu cũng trùng vì mỗi dải chỉ có một cách phủ tối thiểu
print("ví dụ đề: OK (mọi cách hợp lệ + tối thiểu)")

ví dụ đề: OK (mọi cách hợp lệ + tối thiểu)


In [26]:
# Edge case
edge = [
    ("0.0.0.0", 1),               # nhỏ nhất
    ("255.255.255.255", 1),       # địa chỉ cuối cùng của không gian
    ("0.0.0.0", 1000),            # bắt đầu tại 0 (x & -x = 0 phải xử lý riêng)
    ("0.0.0.0", 256),             # đúng một khối /24
    ("10.0.0.0", 512),            # đúng một khối /23
    ("10.0.1.0", 512),            # căn biên /24 chứ không /23 -> cần 2 khối
    ("255.255.255.0", 256),       # cuối không gian, đúng /24
    ("255.255.254.0", 512),       # cuối không gian, đúng /23
    ("255.255.255.250", 6),       # sát cuối không gian
    ("1.2.3.255", 2),             # vượt ranh giới byte: .255 rồi .0 của byte kế
    ("1.2.3.250", 20),            # băng qua ranh giới byte
    ("0.0.0.1", 1000),            # lệch 1 so với biên
    ("128.0.0.0", 1000),          # bit cao nhất bật
]
for ip, n in edge:
    ans = run_all(ip, n)
    print(f"{ip:16s} n={n:4d} -> {len(ans)} khối: {ans[:4]}{' ...' if len(ans) > 4 else ''}")
assert run_all("0.0.0.0", 1) == ["0.0.0.0/32"]
assert run_all("255.255.255.255", 1) == ["255.255.255.255/32"]
assert run_all("0.0.0.0", 256) == ["0.0.0.0/24"]
assert run_all("10.0.1.0", 512) == ["10.0.1.0/24", "10.0.2.0/24"]
assert run_all("1.2.3.255", 2) == ["1.2.3.255/32", "1.2.4.0/32"]
print("edge case: OK")

0.0.0.0          n=   1 -> 1 khối: ['0.0.0.0/32']
255.255.255.255  n=   1 -> 1 khối: ['255.255.255.255/32']
0.0.0.0          n=1000 -> 6 khối: ['0.0.0.0/23', '0.0.2.0/24', '0.0.3.0/25', '0.0.3.128/26'] ...
0.0.0.0          n= 256 -> 1 khối: ['0.0.0.0/24']
10.0.0.0         n= 512 -> 1 khối: ['10.0.0.0/23']
10.0.1.0         n= 512 -> 2 khối: ['10.0.1.0/24', '10.0.2.0/24']
255.255.255.0    n= 256 -> 1 khối: ['255.255.255.0/24']
255.255.254.0    n= 512 -> 1 khối: ['255.255.254.0/23']
255.255.255.250  n=   6 -> 2 khối: ['255.255.255.250/31', '255.255.255.252/30']
1.2.3.255        n=   2 -> 2 khối: ['1.2.3.255/32', '1.2.4.0/32']
1.2.3.250        n=  20 -> 5 khối: ['1.2.3.250/31', '1.2.3.252/30', '1.2.4.0/29', '1.2.4.8/30'] ...
0.0.0.1          n=1000 -> 15 khối: ['0.0.0.1/32', '0.0.0.2/31', '0.0.0.4/30', '0.0.0.8/29'] ...
128.0.0.0        n=1000 -> 6 khối: ['128.0.0.0/23', '128.0.2.0/24', '128.0.3.0/25', '128.0.3.128/26'] ...
edge case: OK


Phản ví dụ cho các cách "gần đúng" người mới hay nghĩ ra:

- *Chỉ nhìn căn biên, quên phần còn lại*: tại `0.0.0.0` với `n = 10` chọn khối `/0` phủ cả không gian.
- *Chỉ nhìn phần còn lại, quên căn biên*: tại `255.0.0.7` với `n = 10` chọn `/29` bắt đầu ở `.7`, thực chất phủ `.0` đến `.7`.
- *Mỗi địa chỉ một `/32`*: phủ đúng nhưng không tối thiểu.

In [27]:
def wrong_only_alignment(ip, n):
    x, res = ip_to_int(ip), []
    while n > 0:
        size = (x & -x) or (1 << 32)               # BUG: quên giới hạn theo n
        res.append(f"{int_to_ip(x)}/{32 - (size.bit_length() - 1)}")
        x += size
        n -= size
    return res

def wrong_only_remaining(ip, n):
    x, res = ip_to_int(ip), []
    while n > 0:
        size = 1 << (n.bit_length() - 1)           # BUG: quên căn biên
        res.append(f"{int_to_ip(x)}/{32 - (size.bit_length() - 1)}")
        x += size
        n -= size
    return res

for name, fn, ip, n in (("chỉ căn biên", wrong_only_alignment, "0.0.0.0", 10),
                        ("chỉ phần còn lại", wrong_only_remaining, "255.0.0.7", 10)):
    ans = fn(ip, n)
    try:
        check_cidr(ip, n, ans)
        print(name, "-> qua kiểm tra?!")
    except AssertionError as e:
        print(f"{name}: {ans} -> bị từ chối: {e}")

chỉ căn biên: ['0.0.0.0/0'] -> bị từ chối: kết thúc sai: 4294967295, mong đợi 9
chỉ phần còn lại: ['255.0.0.7/29', '255.0.0.15/31'] -> bị từ chối: hở hoặc chồng lấn tại 4278190080, mong đợi 4278190087


In [28]:
# Vét cạn: MỌI dải [s, s+n-1] trong miền 6 bit (0..63), tức mọi (start, n) với start + n <= 64, qua cả 3 cách trên số nguyên.
def greedy_blocks(start, n, bits=32):
    res, x, full = [], start, 1 << bits
    while n > 0:
        lowbit = (x & -x) or full
        size = min(lowbit, 1 << (n.bit_length() - 1))
        res.append((x, bits - (size.bit_length() - 1)))
        x += size
        n -= size
    return res

def dp_blocks(start, n, bits=32):
    INF = float("inf")
    f, choice = [INF] * (n + 1), [0] * (n + 1)
    f[n] = 0
    for t in range(n - 1, -1, -1):
        for j in range(bits + 1):
            size = 1 << j
            if size > n - t:
                break
            if (start + t) % size == 0 and f[t + size] + 1 < f[t]:
                f[t], choice[t] = f[t + size] + 1, size
    out, t = [], 0
    while t < n:
        out.append((start + t, bits - (choice[t].bit_length() - 1)))
        t += choice[t]
    return out

BITS = 6
total = 0
max_blocks = 0
for start in range(0, 1 << BITS):
    for n in range(1, (1 << BITS) - start + 1):
        for fn in (greedy_blocks, dp_blocks, cidr_trie_bits):
            blocks = fn(start, n, BITS)
            assert check_blocks(start, n, blocks, BITS)
        max_blocks = max(max_blocks, len(greedy_blocks(start, n, BITS)))
        total += 1
print(f"vét cạn miền {BITS} bit: {total} dải, 3 cách đều hợp lệ và tối thiểu")
print("số khối lớn nhất trong miền", BITS, "bit:", max_blocks, "= 2*bits - 2 =", 2 * BITS - 2)
assert max_blocks == 2 * BITS - 2

vét cạn miền 6 bit: 2080 dải, 3 cách đều hợp lệ và tối thiểu
số khối lớn nhất trong miền 6 bit: 10 = 2*bits - 2 = 10


In [29]:
# Vét cạn trên IP 32 bit thật: byte cuối 0..255 và n = 1..300 (băng qua ranh giới byte)
cnt = 0
for last in range(0, 256, 3):
    ip = f"10.1.2.{last}"
    for n in range(1, 301, 7):
        run_all(ip, n)
        cnt += 1
print("quét", cnt, "ca với IP 32 bit: OK")

# Ngẫu nhiên có seed, IP 32 bit tùy ý, n <= 1000, đảm bảo ip + n - 1 hợp lệ
rng = random.Random(2029)
many_blocks = 0
for t in range(1500):
    n = rng.randint(1, 1000)
    mode = rng.choice(["any", "near_aligned", "near_top", "zero"])
    if mode == "any":
        x = rng.randint(0, (1 << 32) - n)
    elif mode == "near_aligned":
        j = rng.randint(1, 31)
        x = max(0, min(((rng.randint(0, (1 << 32) - 1) >> j) << j) + rng.randint(-5, 5), (1 << 32) - n))
    elif mode == "near_top":
        x = (1 << 32) - n - rng.randint(0, 40)
    else:
        x = rng.randint(0, 40)
    ip = int_to_ip(x)
    ans = run_all(ip, n)
    many_blocks = max(many_blocks, len(ans))
print("1500 ca ngẫu nhiên: OK | số khối lớn nhất đã gặp:", many_blocks, "(<= 2*32-2 = 62)")

quét 3698 ca với IP 32 bit: OK


1500 ca ngẫu nhiên: OK | số khối lớn nhất đã gặp: 16 (<= 2*32-2 = 62)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ giới hạn theo căn biên, quên phần còn lại

Code sai: `size = x & -x` (không cắt theo `n`). Tại `0.0.0.0` bit thấp nhất vô hạn, ta lấy khối `/0` phủ cả không gian: vượt xa `n`. Tại bất kỳ `x` nào, khối căn biên lớn nhất có thể dài hơn phần còn lại.

In [30]:
ans = wrong_only_alignment("0.0.0.0", 10)
print("sai:", ans)
try:
    check_cidr("0.0.0.0", 10, ans)
except AssertionError as e:
    print("bị từ chối:", e)
print("đúng:", Solution().ipToCIDR("0.0.0.0", 10))

sai: ['0.0.0.0/0']
bị từ chối: kết thúc sai: 4294967295, mong đợi 9
đúng: ['0.0.0.0/29', '0.0.0.8/31']


### Lỗi 2 — Chỉ giới hạn theo phần còn lại, quên căn biên

Code sai: `size = 1 << (n.bit_length() - 1)`. Với `255.0.0.7` và `n = 10`, ta chọn khối `8` bắt đầu tại `.7`, nhưng `/29` bắt đầu tại `.7` thực chất phủ `.0` đến `.7` (tiền tố 29 bit của `.7` là `.0`): sai hoàn toàn.

In [31]:
ans = wrong_only_remaining("255.0.0.7", 10)
print("sai:", ans)
lo, hi = block_range(ans[0])
print(f"khối đầu {ans[0]} thực sự phủ {int_to_ip(lo)} .. {int_to_ip(hi)} (không phải từ .7)")
try:
    check_cidr("255.0.0.7", 10, ans)
except AssertionError as e:
    print("bị từ chối:", e)

sai: ['255.0.0.7/29', '255.0.0.15/31']
khối đầu 255.0.0.7/29 thực sự phủ 255.0.0.0 .. 255.0.0.7 (không phải từ .7)
bị từ chối: hở hoặc chồng lấn tại 4278190080, mong đợi 4278190087


### Lỗi 3 — Tính độ dài tiền tố lệch 1

Khối `2^j` địa chỉ có tiền tố `32 - j`, mà `j = size.bit_length() - 1` (không phải `bit_length()`). Viết `32 - size.bit_length()` cho tiền tố lớn hơn 1: mỗi khối bị **thu nhỏ một nửa**.

In [32]:
def wrong_prefix_off_by_one(ip, n):
    x, res = ip_to_int(ip), []
    while n > 0:
        size = min((x & -x) or (1 << 32), 1 << (n.bit_length() - 1))
        k = 32 - size.bit_length()                    # BUG: thiếu "- 1" bên trong
        res.append(f"{int_to_ip(x)}/{k}")
        x += size
        n -= size
    return res

ans = wrong_prefix_off_by_one("255.0.0.7", 10)
print("sai:", ans, "(khối đơn lẻ thành /31 -> phủ 2 địa chỉ; /29 thành /28)")
print("đúng:", ip_to_cidr_greedy("255.0.0.7", 10))
try:
    check_cidr("255.0.0.7", 10, ans)
except AssertionError as e:
    print("bị từ chối:", e)

sai: ['255.0.0.7/31', '255.0.0.8/28', '255.0.0.16/31'] (khối đơn lẻ thành /31 -> phủ 2 địa chỉ; /29 thành /28)
đúng: ['255.0.0.7/32', '255.0.0.8/29', '255.0.0.16/32']
bị từ chối: hở hoặc chồng lấn tại 4278190080, mong đợi 4278190087


### Lỗi 4 — Ghép IP thiếu `& 255`

Khi đổi số về IP, byte giữa phải được cắt `8` bit thấp bằng `& 255`. Nếu chỉ `x >> 16` ta được số lớn hơn `255` (chứa cả byte cao). Với IP như `255.0.0.7` byte đầu ổn nhưng các byte sau sai: dễ lọt qua vài ví dụ nhỏ rồi sai ở IP khác.

In [33]:
def wrong_int_to_ip(x):
    return ".".join(str(x >> s) for s in (24, 16, 8, 0))       # BUG: thiếu & 255

for ip in ("255.0.0.7", "1.2.3.4", "10.20.30.40"):
    print(f"{ip:14s} -> đổi về: {wrong_int_to_ip(ip_to_int(ip))}")
assert wrong_int_to_ip(ip_to_int("1.2.3.4")) != "1.2.3.4"

255.0.0.7      -> đổi về: 255.65280.16711680.4278190087
1.2.3.4        -> đổi về: 1.258.66051.16909060
10.20.30.40    -> đổi về: 10.2580.660510.169090600


### Lỗi 5 — `x & -x` bằng `0` khi `x = 0`: lặp vô hạn

Nếu viết `size = min(x & -x, ...)` mà không xử lý `x = 0`, ta được `size = 0`, `x` và `n` không đổi: vòng `while` chạy mãi. Hãy tái hiện bằng một bộ đếm bảo vệ.

In [34]:
def wrong_zero_case(ip, n, guard=50):
    x, res = ip_to_int(ip), []
    while n > 0 and guard > 0:
        size = min(x & -x, 1 << (n.bit_length() - 1))       # BUG: x = 0 cho size = 0
        k = 32 - (max(size, 1).bit_length() - 1)
        res.append(f"{int_to_ip(x)}/{k}")
        x += size
        n -= size
        guard -= 1
    return res, n, guard

res, n_left, guard_left = wrong_zero_case("0.0.0.0", 10)
print(f"sau 50 vòng vẫn còn n = {n_left}, guard = {guard_left}: lặp vô hạn nếu không có guard; ví dụ khối lặp lại:", res[:2])
print("đúng:", Solution().ipToCIDR("0.0.0.0", 10))
assert n_left == 10 and guard_left == 0

sau 50 vòng vẫn còn n = 10, guard = 0: lặp vô hạn nếu không có guard; ví dụ khối lặp lại: ['0.0.0.0/32', '0.0.0.0/32']
đúng: ['0.0.0.0/29', '0.0.0.8/31']


### Lỗi 6 — Mỗi địa chỉ một khối `/32` (đúng phủ, sai "ít nhất")

Danh sách `n` khối `/32` phủ chính xác dải nhưng không **tối thiểu**. Đề yêu cầu **ít khối nhất**. Mục 5 đã cho thấy 10 khối thay vì 3.

In [35]:
for ip, n, expected in examples:
    singles = ip_to_cidr_singles(ip, n)
    print(f"{ip} n={n}: /32 từng địa chỉ cho {len(singles)} khối, tối thiểu chỉ {len(expected)}")
    assert check_cidr(ip, n, singles, require_min=False)
    assert len(singles) > len(expected)

255.0.0.7 n=10: /32 từng địa chỉ cho 10 khối, tối thiểu chỉ 3
117.145.102.62 n=8: /32 từng địa chỉ cho 8 khối, tối thiểu chỉ 3


## 14. Mẹo & nhận dạng pattern

- **Nhận dạng:** "chia một đoạn số liên tiếp thành ít khối căn biên (lũy thừa của 2)" ⇒ **tham lam với `x & -x`**; cùng họ với phân rã đoạn thành nút **segment tree/Fenwick**.
- Khối CIDR `/k` = đoạn `2^(32-k)` địa chỉ **bắt đầu tại bội của `2^(32-k)`**. Luôn nhớ hai ràng buộc: **căn biên** và **không vượt phần còn lại**.
- `x & -x` cho lũy thừa 2 chia hết `x` lớn nhất; xử lý riêng `x = 0`.
- `1 << (n.bit_length() - 1)` cho lũy thừa 2 lớn nhất `<= n`.
- Đổi IP sang số nguyên rồi làm việc trên số: tránh đụng các ranh giới byte.
- Với bài nhiều đáp án: viết `check_*` kiểm tra tính chất (phủ chính xác + tối thiểu) và đối chiếu một **oracle độc lập** (DP) trên miền nhỏ.
- Thử các **miền nhỏ** (6 bit) để vét cạn mọi dải; tính chất sẽ đúng cho 32 bit.
- Các bài cùng họ: **93. Restore IP Addresses**, **468. Validate IP Address**, **191. Number of 1 Bits**, **231. Power of Two**, **307. Range Sum Query - Mutable** (phân rã đoạn), **1291. Sequential Digits**.

## 15. Biến thể

### 15.1. Đoạn đóng `[lo, hi]` thay vì `(ip, n)`

Chỉ cần đặt `n = hi - lo + 1`. Hàm tổng quát trả về danh sách `(đầu, tiền tố)`.

In [36]:
def range_to_cidr(lo_ip, hi_ip):
    lo, hi = ip_to_int(lo_ip), ip_to_int(hi_ip)
    return [f"{int_to_ip(b)}/{k}" for b, k in greedy_blocks(lo, hi - lo + 1, 32)]

print(range_to_cidr("10.0.0.5", "10.0.0.20"))
print(range_to_cidr("192.168.0.0", "192.168.255.255"))
assert range_to_cidr("192.168.0.0", "192.168.255.255") == ["192.168.0.0/16"]
assert range_to_cidr("0.0.0.0", "255.255.255.255") == ["0.0.0.0/0"]
assert check_cidr("10.0.0.5", 16, range_to_cidr("10.0.0.5", "10.0.0.20"))
print("range_to_cidr OK (đoạn tới cả /16 và cả /0)")

['10.0.0.5/32', '10.0.0.6/31', '10.0.0.8/29', '10.0.0.16/30', '10.0.0.20/32']
['192.168.0.0/16']
range_to_cidr OK (đoạn tới cả /16 và cả /0)


### 15.2. Chiều ngược lại: CIDR → đoạn địa chỉ, và "IP có thuộc khối không?"

Ta đã dùng `block_range`. Kiểm tra thuộc khối: `lo <= ip_int <= hi`. Hoặc dùng mặt nạ: hai số cùng `k` bit đầu khi `(a ^ b) >> (32 - k) == 0`.

In [37]:
def in_block(ip, cidr):
    lo, hi = block_range(cidr)
    return lo <= ip_to_int(ip) <= hi

def in_block_mask(ip, cidr):
    base, k = parse_cidr(cidr)
    return k == 0 or ((ip_to_int(ip) ^ base) >> (32 - k)) == 0

rng = random.Random(5)
for _ in range(2000):
    x = rng.randint(0, (1 << 32) - 1)
    k = rng.randint(0, 32)
    base = x if rng.random() < 0.5 else rng.randint(0, (1 << 32) - 1)
    cidr = f"{int_to_ip(base)}/{k}"
    probe = int_to_ip(x ^ (rng.randint(0, 3) << rng.randint(0, 31)) & 0xFFFFFFFF)
    assert in_block(probe, cidr) == in_block_mask(probe, cidr)
print(in_block("255.0.0.12", "255.0.0.8/29"), in_block("255.0.0.7", "255.0.0.8/29"))

True False


### 15.3. Số khối tối đa là bao nhiêu? (với `bits` bit)

Thí nghiệm ở mục 12 cho thấy trên miền `b` bit số khối lớn nhất là `2b - 2`; với 32 bit đó là `62`. Cận này đạt được khi dải bắt đầu ngay sau một biên lớn và kết thúc ngay trước một biên lớn khác, ví dụ `[1, 2^b - 2]`.

In [38]:
for bits in (3, 4, 5, 6, 8):
    n = (1 << bits) - 2
    blocks = greedy_blocks(1, n, bits)
    assert len(blocks) == 2 * bits - 2
    print(f"bits = {bits}: dải [1, {n}] cần {len(blocks)} khối = 2*{bits} - 2; kích thước các khối:",
          [1 << (bits - k) for _, k in blocks])
blocks32 = greedy_blocks(1, (1 << 32) - 2, 32)
assert len(blocks32) == 62
print("32 bit: dải [1, 2^32 - 2] cần", len(blocks32), "khối (cận tối đa)")

bits = 3: dải [1, 6] cần 4 khối = 2*3 - 2; kích thước các khối: [1, 2, 2, 1]
bits = 4: dải [1, 14] cần 6 khối = 2*4 - 2; kích thước các khối: [1, 2, 4, 4, 2, 1]
bits = 5: dải [1, 30] cần 8 khối = 2*5 - 2; kích thước các khối: [1, 2, 4, 8, 8, 4, 2, 1]
bits = 6: dải [1, 62] cần 10 khối = 2*6 - 2; kích thước các khối: [1, 2, 4, 8, 16, 16, 8, 4, 2, 1]
bits = 8: dải [1, 254] cần 14 khối = 2*8 - 2; kích thước các khối: [1, 2, 4, 8, 16, 32, 64, 64, 32, 16, 8, 4, 2, 1]
32 bit: dải [1, 2^32 - 2] cần 62 khối (cận tối đa)


Dãy kích thước đúng như đã dự đoán: tăng dần `1, 2, 4, ...` rồi giảm dần `..., 4, 2, 1`.

### 15.4. IPv6 (128 bit) và độ rộng tùy ý

Thuật toán không phụ thuộc vào con số 32: chỉ cần thay `32` bằng `bits`. Python có số nguyên lớn tùy ý nên chạy ngay cho `bits = 128`.

In [39]:
def check_blocks_wide(start, n, blocks, bits):
    return check_blocks(start, n, blocks, bits, require_min=False)

rng = random.Random(7)
for _ in range(300):
    bits = rng.choice([8, 16, 32, 64, 128])
    n = rng.randint(1, min(1000, 1 << bits))
    start = rng.randint(0, (1 << bits) - n)
    blocks = greedy_blocks(start, n, bits)
    assert check_blocks_wide(start, n, blocks, bits)
    assert len(blocks) <= max(1, 2 * bits - 2)
    if bits <= 32:
        assert len(blocks) == len(cidr_trie_bits(start, n, bits))
print("bits tùy ý (8..128): phủ chính xác và đúng cận 2*bits - 2")
print(len(greedy_blocks(1, (1 << 128) - 2, 128)), "khối cho dải [1, 2^128 - 2]")

bits tùy ý (8..128): phủ chính xác và đúng cận 2*bits - 2
254 khối cho dải [1, 2^128 - 2]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. `/32` từng địa chỉ | Mỗi IP một khối | `O(n)` | `O(n)` | Chỉ để thấy "đúng phủ nhưng không tối thiểu" |
| 2. Tham lam (nộp) | Khối lớn nhất căn biên và vừa phần còn lại: `min(x & -x, 2^⌊log2 n⌋)` | `O(B)`, `B <= 62` | `O(B)` | Mặc định: ngắn, nhanh, bit trick |
| 3. Quy hoạch động | `f[t] = 1 + min f[t + size]` trên mọi khối hợp lệ | `O(n log n)` | `O(n)` | Làm oracle độc lập; không cần chứng minh tham lam |
| 4. Cây nhị phân tiền tố | Nút nằm trọn thì lấy, giao một phần thì chia đôi | `O(32 · B)` | `O(32 + B)` | Khi nghĩ theo segment tree; tổng quát cho mọi độ rộng |

Cách 2 và 4 cho đúng cùng danh sách (đã kiểm chứng); cách 3 cho cùng số khối.

## 17. Bài tập vận dụng

Làm tuần tự từ dễ đến khó. Các cell có chữ `TODO` **không được gọi** khi chạy notebook; bạn tự điền rồi bỏ dấu `#` ở các dòng `assert` để kiểm tra. Đáp án nằm ở mục 19.

### Bài 1 — Đoán output (dễ)

Không chạy code, hãy đoán output của ba dòng sau, rồi chạy để kiểm tra:

```python
print(ip_to_int("0.0.1.2"))
print(12 & -12)
print(1 << (13).bit_length() - 1)
```

In [40]:
# Bài 1: viết dự đoán của bạn vào biến bên dưới (chưa chạy code mẫu)
my_guess_line1 = None
my_guess_line2 = None
my_guess_line3 = None

### Bài 2 — Tính tay (dễ-vừa)

Dùng tham lam, hãy tính tay `ipToCIDR("0.0.0.5", 6)` (phủ `.5` đến `.10`): với mỗi bước ghi `x` (byte cuối), `x & -x`, `n` còn, `size`, tiền tố, rồi viết danh sách khối.

In [41]:
# Bài 2: điền kết quả tính tay
hand_blocks = None       # list các chuỗi CIDR theo thứ tự
hand_num_blocks = None   # số khối

### Bài 3 — Điền chỗ trống (vừa)

Hoàn thành hàm theo Cách 2 (tham lam), dùng sẵn `ip_to_int` và `int_to_ip`.

In [42]:
def exercise_3_greedy(ip: str, n: int) -> List[str]:
    x = ip_to_int(ip)
    ans = []
    # TODO 1: while n > 0
    #     TODO 2: lowbit = x & -x (nếu x == 0 thì dùng 1 << 32)
    #     TODO 3: size = min(lowbit, lũy thừa 2 lớn nhất <= n)   (dùng bit_length)
    #     TODO 4: k = 32 - (size.bit_length() - 1); thêm f"{int_to_ip(x)}/{k}" vào ans
    #     TODO 5: x += size; n -= size
    return None


# for ip, n, expected in examples:
#     assert exercise_3_greedy(ip, n) == expected
# assert exercise_3_greedy("0.0.0.0", 256) == ["0.0.0.0/24"]

### Bài 4 — Tự viết lại (vừa-khó)

Không nhìn lại Cách 4, hãy tự viết `exercise_4_trie(ip, n)` bằng **đệ quy trên cây nhị phân tiền tố**: nút không giao dải thì bỏ, nút nằm trọn thì lấy, ngược lại chia đôi. Kết quả phải **đúng thứ tự tăng dần** theo địa chỉ.

In [43]:
def exercise_4_trie(ip: str, n: int) -> List[str]:
    # TODO
    pass


# for ip, n, expected in examples:
#     assert exercise_4_trie(ip, n) == expected
# assert exercise_4_trie("10.0.1.0", 512) == ["10.0.1.0/24", "10.0.2.0/24"]

### Bài 5 — Mini challenge: gộp danh sách CIDR (khó)

Cho danh sách các khối CIDR **không chồng lấn**. Hãy trả về danh sách CIDR **ít khối nhất** phủ **chính xác cùng tập địa chỉ**. Gợi ý: đổi mỗi khối thành đoạn `[đầu, cuối]`, sắp xếp, **gộp các đoạn liền kề** (đầu của đoạn sau = cuối đoạn trước + 1), rồi với mỗi đoạn gộp dùng lại tham lam.

```python
exercise_5_aggregate(["10.0.0.0/25", "10.0.0.128/25"]) -> ["10.0.0.0/24"]
exercise_5_aggregate(["1.1.1.0/32", "1.1.1.1/32", "1.1.1.2/31"]) -> ["1.1.1.0/30"]
exercise_5_aggregate(["10.0.0.0/24", "10.0.2.0/24"]) -> ["10.0.0.0/24", "10.0.2.0/24"]   # có khoảng trống
```

In [44]:
def exercise_5_aggregate(blocks: List[str]) -> List[str]:
    # TODO
    pass


# assert exercise_5_aggregate(["10.0.0.0/25", "10.0.0.128/25"]) == ["10.0.0.0/24"]
# assert exercise_5_aggregate(["1.1.1.0/32", "1.1.1.1/32", "1.1.1.2/31"]) == ["1.1.1.0/30"]
# assert exercise_5_aggregate(["10.0.0.0/24", "10.0.2.0/24"]) == ["10.0.0.0/24", "10.0.2.0/24"]

## 18. Gợi ý 3 mức

**Bài 1**
- Mức 1: `ip_to_int` ghép `a<<24 | b<<16 | c<<8 | d`; `x & -x` giữ bit `1` thấp nhất; `bit_length` đếm số bit.
- Mức 2: `0.0.1.2 = 1*256 + 2`; `12 = 1100` nên bit thấp nhất là `4`; `13 = 1101` có 4 bit nên `1 << 3`.
- Mức 3: `258`, `4`, `8`.

**Bài 2**
- Mức 1: byte cuối `5 = 101`: bit thấp nhất `1`.
- Mức 2: các bước: `x=5` (lowbit 1, n=6 → size 1), `x=6` (lowbit 2, n=5 → pow2 4, size 2), `x=8` (lowbit 8, n=3 → pow2 2, size 2), `x=10` (lowbit 2, n=1 → size 1).
- Mức 3: `["0.0.0.5/32", "0.0.0.6/31", "0.0.0.8/31", "0.0.0.10/32"]`, `4` khối.

**Bài 3**
- Mức 1: giống mục 7.1.
- Mức 2: `size = min(lowbit, 1 << (n.bit_length() - 1))`.
- Mức 3: nhớ xử lý `x == 0` bằng `or (1 << 32)`, và `k = 32 - (size.bit_length() - 1)`.

**Bài 4**
- Mức 1: hàm con `go(node_start, level)` với nút phủ `[node_start, node_start + 2^level - 1]`, gọi `go(0, 32)`.
- Mức 2: ngoài dải thì `return`; nằm trọn thì thêm `f"{int_to_ip(node_start)}/{32 - level}"` rồi `return`.
- Mức 3: ngược lại gọi `go(node_start, level - 1)` rồi `go(node_start + (1 << (level - 1)), level - 1)` (trái trước phải sau).

**Bài 5**
- Mức 1: dùng `block_range` để lấy `[đầu, cuối]` của mỗi khối rồi sắp xếp theo đầu.
- Mức 2: duyệt, nếu `đầu_mới == cuối_hiện_tại + 1` thì kéo dài đoạn hiện tại, ngược lại chốt đoạn cũ và bắt đầu đoạn mới.
- Mức 3: với mỗi đoạn gộp `[lo, hi]` gọi `greedy_blocks(lo, hi - lo + 1, 32)` rồi đổi sang chuỗi.

## 19. Đáp án tham khảo

### Đáp án Bài 1

In [45]:
ans1 = [ip_to_int("0.0.1.2"), 12 & -12, 1 << (13).bit_length() - 1]
print(ans1)
assert ans1 == [258, 4, 8]

[258, 4, 8]


### Đáp án Bài 2

In [46]:
ans2_blocks = ["0.0.0.5/32", "0.0.0.6/31", "0.0.0.8/31", "0.0.0.10/32"]
ans2_num = 4
assert Solution().ipToCIDR("0.0.0.5", 6) == ans2_blocks
assert check_cidr("0.0.0.5", 6, ans2_blocks)
assert len(ans2_blocks) == ans2_num == min_blocks_dp(5, 6)
print("Bài 2 OK")

Bài 2 OK


### Đáp án Bài 3

In [47]:
def ans3_greedy(ip: str, n: int) -> List[str]:
    x = ip_to_int(ip)
    ans = []
    while n > 0:
        lowbit = (x & -x) or (1 << 32)
        size = min(lowbit, 1 << (n.bit_length() - 1))
        k = 32 - (size.bit_length() - 1)
        ans.append(f"{int_to_ip(x)}/{k}")
        x += size
        n -= size
    return ans

for ip, n, expected in examples:
    assert ans3_greedy(ip, n) == expected
assert ans3_greedy("0.0.0.0", 256) == ["0.0.0.0/24"]
rng = random.Random(41)
for _ in range(300):
    n = rng.randint(1, 1000)
    ip = int_to_ip(rng.randint(0, (1 << 32) - n))
    assert check_cidr(ip, n, ans3_greedy(ip, n))
print("Bài 3 OK")

Bài 3 OK


### Đáp án Bài 4

In [48]:
def ans4_trie(ip: str, n: int) -> List[str]:
    lo = ip_to_int(ip)
    hi = lo + n - 1
    res = []

    def go(node_start: int, level: int) -> None:
        node_end = node_start + (1 << level) - 1
        if node_end < lo or node_start > hi:
            return
        if lo <= node_start and node_end <= hi:
            res.append(f"{int_to_ip(node_start)}/{32 - level}")
            return
        half = 1 << (level - 1)
        go(node_start, level - 1)
        go(node_start + half, level - 1)

    go(0, 32)
    return res

for ip, n, expected in examples:
    assert ans4_trie(ip, n) == expected
assert ans4_trie("10.0.1.0", 512) == ["10.0.1.0/24", "10.0.2.0/24"]
rng = random.Random(42)
for _ in range(300):
    n = rng.randint(1, 1000)
    ip = int_to_ip(rng.randint(0, (1 << 32) - n))
    res = ans4_trie(ip, n)
    assert check_cidr(ip, n, res)
    assert res == ans3_greedy(ip, n)
print("Bài 4 OK")

Bài 4 OK


### Đáp án Bài 5

In [49]:
def ans5_aggregate(blocks: List[str]) -> List[str]:
    intervals = sorted(block_range(b) for b in blocks)
    merged = []
    for lo, hi in intervals:
        if merged and lo == merged[-1][1] + 1:
            merged[-1][1] = hi
        else:
            merged.append([lo, hi])
    out = []
    for lo, hi in merged:
        out.extend(f"{int_to_ip(b)}/{k}" for b, k in greedy_blocks(lo, hi - lo + 1, 32))
    return out

assert ans5_aggregate(["10.0.0.0/25", "10.0.0.128/25"]) == ["10.0.0.0/24"]
assert ans5_aggregate(["1.1.1.0/32", "1.1.1.1/32", "1.1.1.2/31"]) == ["1.1.1.0/30"]
assert ans5_aggregate(["10.0.0.0/24", "10.0.2.0/24"]) == ["10.0.0.0/24", "10.0.2.0/24"]

# Kiểm chứng ngẫu nhiên: cắt một dải thành các khối /32 và /31 rồi xáo trộn, gộp lại phải bằng đáp án tối thiểu của cả dải
rng = random.Random(43)
for _ in range(200):
    n = rng.randint(1, 200)
    start = rng.randint(0, 5000)
    pieces = [f"{int_to_ip(start + i)}/32" for i in range(n)]
    rng.shuffle(pieces)
    got = ans5_aggregate(pieces)
    assert check_cidr(int_to_ip(start), n, got)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

### Template

```python
class Solution:
    def ipToCIDR(self, ip: str, n: int) -> List[str]:
        a, b, c, d = map(int, ip.split("."))
        x = (a << 24) | (b << 16) | (c << 8) | d          # IP -> số nguyên 32 bit
        ans = []
        while n > 0:
            lowbit = (x & -x) or (1 << 32)                 # khối căn biên lớn nhất tại x (x = 0: vô hạn)
            size = min(lowbit, 1 << (n.bit_length() - 1))  # và không vượt phần còn lại
            k = 32 - (size.bit_length() - 1)               # độ dài tiền tố
            ans.append(f"{(x >> 24) & 255}.{(x >> 16) & 255}.{(x >> 8) & 255}.{x & 255}/{k}")
            x += size
            n -= size
        return ans
```

Nhớ ba ý: **khối `/k` có `2^(32-k)` địa chỉ và phải bắt đầu tại bội của kích thước**, **kích thước lớn nhất = `min(x & -x, 2^⌊log2 n⌋)`**, **bài nhiều đáp án thì test bằng `check_cidr` (phủ chính xác + tối thiểu)**.

### Checklist trước khi submit

- [ ] Đã đổi IP sang số nguyên 32 bit và đổi ngược có `& 255` cho từng byte.
- [ ] Kích thước khối bị chặn bởi **cả hai**: căn biên (`x & -x`) và phần còn lại (`2^⌊log2 n⌋`).
- [ ] Xử lý `x = 0` (`x & -x = 0`) bằng `or (1 << 32)`.
- [ ] Độ dài tiền tố `k = 32 - (size.bit_length() - 1)` (không lệch 1).
- [ ] Đã thử `n = 1`, `0.0.0.0`, `255.255.255.255`, dải băng qua ranh giới byte, dải đúng một khối.
- [ ] Test bằng hàm kiểm tra phủ đúng và số khối tối thiểu, không so cứng chuỗi.
- [ ] Time `O(log)` số khối (tối đa ~62), Space `O(số khối)`.
- [ ] (Premium) đã cân nhắc đối chiếu lại đề gốc nếu có tài khoản.